# Load Trained SU(2) Field Emulators (TF + PyTorch)

This notebook only loads and runs previously trained networks.

Model mapping:
- **`phi` model** = **Question 1 / `nn1`** (predicts $\Phi(x) \in \mathbb{R}^4$).
- **`W` model** = **`nn2`** (predicts flattened $W_\mu^a(x) \in \mathbb{R}^{12}$).

Training input domain used originally:
- $x=(t,x,y,z)$ sampled uniformly from **$[-5.0, 5.0]^4$**.

Inputs outside this range are extrapolation and may have higher error.


In [2]:
import os
import numpy as np
# import tensorflow as tf
import torch
from torch import nn

SEED = 1234
np.random.seed(SEED)
# tf.random.set_seed(SEED)
torch.manual_seed(SEED)

DTYPE = np.float32

# print('TensorFlow version:', tf.__version__)
print('PyTorch version:', torch.__version__)


PyTorch version: 2.11.0


## Paths and Training Range Metadata


In [10]:
TORCH_PHI_WEIGHTS = 'weights_torch_phi.pt'     # nn1 / question 1
TORCH_W_WEIGHTS = 'weights_torch_W.pt'         # nn2

# Original train range metadata
TRAIN_RANGE_MIN = -5.0
TRAIN_RANGE_MAX = 5.0

print('Training range per coordinate:', (TRAIN_RANGE_MIN, TRAIN_RANGE_MAX))
print('Expected input shape: (N, 4) with columns [t, x, y, z]')

for p in [TORCH_PHI_WEIGHTS, TORCH_W_WEIGHTS]:
    print(f'{p}:', 'FOUND' if os.path.exists(p) else 'MISSING')


Training range per coordinate: (-5.0, 5.0)
Expected input shape: (N, 4) with columns [t, x, y, z]
weights_torch_phi.pt: FOUND
weights_torch_W.pt: FOUND


## PyTorch Loader
Architecture matches training:
- Input 4 -> hidden [128,128,128] ReLU -> output 4 (`phi`/`nn1`) or 12 (`W`/`nn2`).


In [5]:
class TorchMLP(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, output_dim),
        )

    def forward(self, x):
        return self.net(x)

# nn1 / question 1: phi model
torch_phi = TorchMLP(output_dim=4)
torch_phi.load_state_dict(torch.load(TORCH_PHI_WEIGHTS, map_location='cpu'))
torch_phi.eval()

# nn2: W model
torch_W = TorchMLP(output_dim=12)
torch_W.load_state_dict(torch.load(TORCH_W_WEIGHTS, map_location='cpu'))
torch_W.eval()

print('Loaded PyTorch phi (nn1/question1) and W (nn2) models.')


Loaded PyTorch phi (nn1/question1) and W (nn2) models.


## Inference Helpers
`predict_phi_*` returns shape `(N,4)` and `predict_W_*` returns shape `(N,12)` (flattened $(\mu,a)$).


In [9]:
def check_range(X):
    x_min = float(np.min(X))
    x_max = float(np.max(X))
    in_range = (x_min >= TRAIN_RANGE_MIN) and (x_max <= TRAIN_RANGE_MAX)
    print(f'Input range observed: [{x_min:.3f}, {x_max:.3f}]')
    if not in_range:
        print('Warning: some points are outside training range [-5, 5].')

def predict_phi_torch(X):
    X = np.asarray(X, dtype=DTYPE)
    check_range(X)
    with torch.no_grad():
        out = torch_phi(torch.from_numpy(X))
    return out.numpy()

def predict_W_torch(X):
    X = np.asarray(X, dtype=DTYPE)
    check_range(X)
    with torch.no_grad():
        out = torch_W(torch.from_numpy(X))
    return out.numpy()


## Example Usage
The same input points can be evaluated with either framework.


In [11]:
X_new = np.array([
    [0.0, 0.0, 0.0, 0.0],
    [1.0, -2.0, 0.5, 3.0],
    [-4.2, 1.3, -0.7, 2.1],
], dtype=DTYPE)

phi_torch_pred = predict_phi_torch(X_new)
W_torch_pred = predict_W_torch(X_new)

print('phi_torch_pred shape:', phi_torch_pred.shape, '  (nn1/question1)')
print('W_torch_pred shape:  ', W_torch_pred.shape, '  (nn2)')

print('\nFirst sample (Torch):')
print('  phi:', phi_torch_pred[0])
print('  W (first 6 of 12):', W_torch_pred[0, :6])


Input range observed: [-4.200, 3.000]
Input range observed: [-4.200, 3.000]
phi_torch_pred shape: (3, 4)   (nn1/question1)
W_torch_pred shape:   (3, 12)   (nn2)

First sample (Torch):
  phi: [ 1.0138465  -0.01207616  0.00967196 -0.01923794]
  W (first 6 of 12): [ 0.07362024  0.0177945  -0.00834646 -0.00796714 -0.03773176 -0.10067789]


## Notes
- `phi` = **nn1 / question 1** target with 4 outputs.
- `W` = **nn2** target with 12 outputs (flattened $(\mu,a)$).
- Training range was $[-5,5]^4$ for $(t,x,y,z)$.
